# 00 - Préparer EXIOBASE 3.10.2

Ce notebook prépare les données qui seront utilisées dans la suite du parcours. À la fin, vous saurez :

- identifier précisément la version et le millésime d’EXIOBASE utilisés ;
- télécharger une table industrie-par-industrie ;
- contrôler l’intégrité du fichier téléchargé ;
- charger la table avec PyMRIO ;
- vérifier que sa structure est celle attendue.

> Ce notebook ne calcule pas encore d’empreinte environnementale. Les matrices économiques seront expliquées dans le notebook `01`, puis les comptes environnementaux dans le notebook `02`.

## 1. Préparer l’environnement Python

Depuis un terminal ouvert dans le dossier du projet, l’environnement peut être créé avec :

```bash
uv venv
uv pip install pymrio==0.6.3 pandas==3.0.5 jupyterlab==4.6.3
uv run jupyter lab
```

Les versions sont indiquées afin que tous les étudiants travaillent dans le même environnement.

In [1]:
from hashlib import md5
from pathlib import Path
from urllib.request import urlretrieve

import pandas as pd
import pymrio as mr

print(f"PyMRIO : {mr.__version__}")
print(f"pandas  : {pd.__version__}")

PyMRIO : 0.6.3
pandas  : 3.0.5


## 2. Identifier les données utilisées

Nous utilisons **EXIOBASE 3.10.2**, publié en mai 2026 :

- dépôt et licence : <https://zenodo.org/records/20051562> ;
- DOI de cette version : <https://doi.org/10.5281/zenodo.20051562> ;
- millésime retenu : **2022** ;
- format retenu : **industrie par industrie** (`ixi`).

Le millésime désigne l’année représentée par la table ; il ne faut pas le confondre avec la version de la base. EXIOBASE 3.10.2 contient des tables allant jusqu’en 2024, mais 2022 est ici privilégiée car les données économiques, énergétiques et d’émissions ont été mises à jour jusqu’à cette année. Les années ultérieures reposent davantage sur des prolongements statistiques (*now-casting*).

### Licence et citation

EXIOBASE 3.10.2 est diffusée sous une licence spécifique, dérivée de CC BY-SA-NC pour les usages académiques non commerciaux. Avant toute diffusion de résultats, il faut lire la licence du dépôt Zenodo. Elle autorise la publication de résultats agrégés avec attribution, mais encadre notamment la redistribution des données brutes et des coefficients dérivés.

Pour un rapport ou un poster, il faut au minimum mentionner :

> EXIOBASE 3, version 3.10.2, DOI : 10.5281/zenodo.20051562 ; Stadler et al. (2018), *Journal of Industrial Ecology*, DOI : 10.1111/jiec.12715.

PyMRIO doit également être cité lorsqu’il est utilisé pour effectuer les calculs : Stadler (2021), DOI : 10.5334/jors.251.

## 3. Choisir le dossier et la table

Une table `ixi` décrit les transactions entre **industries**. Une table `pxp` décrit les transactions entre **produits**. Le parcours utilise `ixi`, qui comporte 163 industries et convient bien à une analyse sectorielle nationale.

La cellule suivante centralise tous les paramètres. Le dossier proposé est commun aux autres projets qui utilisent déjà EXIOBASE sur cette machine.

In [2]:
VERSION_EXIOBASE = "3.10.2"
DOI_EXIOBASE = "10.5281/zenodo.20051562"
IDENTIFIANT_ZENODO = "20051562"
ANNEE = 2022
SYSTEME = "ixi"

DOSSIER_EXIOBASE = Path("D:/EXIOBASE/3.10.2")
NOM_ARCHIVE = f"IOT_{ANNEE}_{SYSTEME}.zip"
ARCHIVE_EXIOBASE = DOSSIER_EXIOBASE / NOM_ARCHIVE
URL_ARCHIVE = (
    f"https://zenodo.org/records/{IDENTIFIANT_ZENODO}/files/"
    f"{NOM_ARCHIVE}?download=1"
)

DOSSIER_EXIOBASE.mkdir(parents=True, exist_ok=True)
ARCHIVE_EXIOBASE

WindowsPath('D:/EXIOBASE/3.10.2/IOT_2022_ixi.zip')

## 4. Télécharger l’archive

L’archive 2022 `ixi` pèse environ 243 Mo. Le téléchargement n’est effectué que si elle n’existe pas déjà. Un fichier portant l’extension `.part` est utilisé pendant le transfert afin qu’un téléchargement interrompu ne soit pas confondu avec une archive complète.

Nous employons ici le lien direct publié par Zenodo. Dans l’environnement testé, la fonction de téléchargement automatique de PyMRIO 0.6.3 ne détecte pas correctement les fichiers du dépôt 3.10.2. PyMRIO sera bien utilisé ensuite pour lire et calculer la table.

Si le téléchargement automatique échoue, le fichier peut être téléchargé manuellement depuis <https://zenodo.org/records/20051562> puis placé dans le dossier indiqué ci-dessus.

In [3]:
def afficher_progression(nombre_blocs, taille_bloc, taille_totale):
    """Affiche l’avancement du téléchargement sur une seule ligne."""
    if taille_totale <= 0:
        return
    proportion = min(nombre_blocs * taille_bloc / taille_totale, 1)
    print(f"\rTéléchargement : {proportion:6.1%}", end="")


if ARCHIVE_EXIOBASE.exists():
    print(f"Archive déjà présente : {ARCHIVE_EXIOBASE}")
else:
    archive_partielle = ARCHIVE_EXIOBASE.with_suffix(".zip.part")
    print(f"Téléchargement depuis {URL_ARCHIVE}")
    urlretrieve(URL_ARCHIVE, archive_partielle, reporthook=afficher_progression)
    archive_partielle.replace(ARCHIVE_EXIOBASE)
    print(f"\nArchive enregistrée : {ARCHIVE_EXIOBASE}")

Archive déjà présente : D:\EXIOBASE\3.10.2\IOT_2022_ixi.zip


## 5. Vérifier l’intégrité de l’archive

Une somme de contrôle est une courte empreinte numérique d’un fichier. Si un seul octet change, la somme change également. Zenodo publie la somme MD5 suivante pour `IOT_2022_ixi.zip` :

```text
7085d04f8ec0cf40f1549c878b24bc41
```

La comparaison permet de repérer un téléchargement incomplet ou corrompu.

In [4]:
MD5_ATTENDU = "7085d04f8ec0cf40f1549c878b24bc41"


def calculer_md5(chemin, taille_bloc=1024 * 1024):
    """Calcule la somme MD5 d’un fichier sans le charger en mémoire."""
    somme = md5()
    with chemin.open("rb") as fichier:
        for bloc in iter(lambda: fichier.read(taille_bloc), b""):
            somme.update(bloc)
    return somme.hexdigest()


md5_obtenu = calculer_md5(ARCHIVE_EXIOBASE)
if md5_obtenu != MD5_ATTENDU:
    raise ValueError(
        "La somme MD5 ne correspond pas à celle publiée par Zenodo. "
        "L’archive doit être téléchargée à nouveau."
    )

print(f"Archive valide, MD5 : {md5_obtenu}")

Archive valide — MD5 : 7085d04f8ec0cf40f1549c878b24bc41


## 6. Charger EXIOBASE avec PyMRIO

PyMRIO peut lire directement l’archive ZIP : il ne faut pas la décompresser. Le chargement peut prendre quelques minutes.

In [5]:
io = mr.parse_exiobase3(path=ARCHIVE_EXIOBASE)
print("Table EXIOBASE chargée.")

Table EXIOBASE chargée.


## 7. Contrôler la structure chargée

EXIOBASE 3.10.2 distingue le cœur économique des extensions environnementales et sociales. Pour une table `ixi`, nous attendons notamment :

- 49 régions : 44 pays et 5 régions « reste du monde » ;
- 163 industries ;
- les extensions `air_emissions`, `material`, `water` et `factor_inputs`.

Les assertions ci-dessous arrêtent immédiatement le notebook si la table chargée ne correspond pas à ces choix.

In [6]:
regions = list(io.get_regions())
secteurs = list(io.get_sectors())
extensions = list(io.get_extensions())
extensions_indispensables = {
    "air_emissions",
    "factor_inputs",
    "material",
    "water",
}

assert len(regions) == 49, f"49 régions attendues, {len(regions)} obtenues"
assert len(secteurs) == 163, f"163 industries attendues, {len(secteurs)} obtenues"
assert extensions_indispensables.issubset(extensions), (
    "Certaines extensions attendues sont absentes : "
    f"{extensions_indispensables.difference(extensions)}"
)

pd.Series(
    {
        "Version demandée": VERSION_EXIOBASE,
        "Millésime": ANNEE,
        "Système": SYSTEME,
        "Nombre de régions": len(regions),
        "Nombre d’industries": len(secteurs),
        "Nombre d’extensions": len(extensions),
    },
    name="Caractéristiques de la table",
)

Version demandée       3.10.2
Millésime                2022
Système                   ixi
Nombre de régions          49
Nombre d’industries       163
Nombre d’extensions         8
Name: Caractéristiques de la table, dtype: object

In [7]:
pd.DataFrame(
    {
        "extension PyMRIO": extensions,
        "rôle": [
            {
                "air_emissions": "émissions atmosphériques",
                "energy": "utilisation d’énergie",
                "employment": "emploi et heures travaillées",
                "factor_inputs": "valeur ajoutée et facteurs de production",
                "land": "utilisation des sols",
                "material": "extraction de matières",
                "nutrients": "rejets d’azote et de phosphore",
                "water": "prélèvements et consommation d’eau",
            }.get(extension, "autre extension")
            for extension in extensions
        ],
    }
)

,extension PyMRIO,rôle
0,nutrients,rejets d’azote et de phosphore
1,air_emissions,émissions atmosphériques
2,factor_inputs,valeur ajoutée et facteurs de production
3,energy,utilisation d’énergie
4,water,prélèvements et consommation d’eau
5,employment,emploi et heures travaillées
6,land,utilisation des sols
7,material,extraction de matières


## 8. Calculer les tables dérivées

L’archive contient les données économiques et les pressions directes. Les coefficients, multiplicateurs et comptes d’empreinte doivent ensuite être calculés. `calc_all()` effectue ces opérations pour le cœur économique et pour chaque extension.

Cette étape est la plus exigeante en mémoire et peut prendre plusieurs minutes. Les objets créés, notamment l’inverse de Leontief, seront expliqués dans les notebooks suivants.

In [8]:
io.calc_all()

tables_attendues = {"A", "L", "Z", "Y", "x"}
tables_disponibles = {nom for nom, valeur in vars(io).items() if isinstance(valeur, pd.DataFrame)}
assert tables_attendues.issubset(tables_disponibles)
assert io.air_emissions.D_cba is not None
assert io.air_emissions.D_pba is not None

print("Calcul terminé : la table est prête pour les notebooks 01 et 02.")

Calcul terminé : la table est prête pour les notebooks 01 et 02.


## À retenir

- **3.10.2** est la version de la base ; **2022** est le millésime de la table.
- `ixi` signifie que les lignes et colonnes du cœur économique représentent des industries.
- l’archive reste hors du dépôt Git et peut être partagée entre plusieurs projets ;
- PyMRIO charge le cœur économique et les extensions dans un même objet `io` ;
- les résultats doivent toujours être accompagnés de la version, du millésime, du système (`ixi` ou `pxp`) et de la licence utilisés.

Le notebook `01` partira de cet objet pour expliquer comment lire les matrices économiques.